## Phase 1

In [11]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# %% [1] Environment Imports & Setup
import os
import shutil
import sys
import gymnasium as gym
import torch

sys.path.insert(0, os.path.abspath(".."))

from src.rl_2.env_adapter import MatchEnv
from src.rl_2.model import ActorCritic
from src.rl_2.pool import PoolOpponentController
from src.rl_2.ppo import train_mappo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


TEAM_SIZE = 2
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0  
ROUND_STEPS_S2 = 3600 
NUM_ENVS = 24 

SAVE_DIR_S2 = "models/stage2/phase1"
POOL_DIR_S2 = os.path.join(SAVE_DIR_S2, "pool")
os.makedirs(POOL_DIR_S2, exist_ok=True)

Using device: cuda


In [3]:
# Seed Opponent Pool with Stage 1 Champion
STAGE1_BEST = "models/stage1/phase4/best_model.pt"
if os.path.exists(STAGE1_BEST):
  shutil.copy(STAGE1_BEST, os.path.join(POOL_DIR_S2, "champion.pt"))
  shutil.copy(STAGE1_BEST, os.path.join(POOL_DIR_S2, "history_0.pt"))
  print(f"✅ Initialized Stage 2 pool with Stage 1 Champion weights.")
else:
  raise FileNotFoundError(f"Missing Stage 1 model at: {STAGE1_BEST}")

def make_s2_env(env_rank: int):

  def _thunk():
    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S2,
        team="blue",
        device="cpu",
        p_random=0.05,
        p_heuristic=0.55,
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team="red",
        max_round_steps=ROUND_STEPS_S2,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
    )
    env.reset(seed=5000 + env_rank)
    return env

  return _thunk


envs_s2 = gym.vector.AsyncVectorEnv(
    [make_s2_env(i) for i in range(NUM_ENVS)],
    context="fork",
)


model_s2 = ActorCritic().to(device)
ckpt = torch.load(STAGE1_BEST, map_location=device, weights_only=False)
state_dict = (
    ckpt["model_state_dict"]
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt
    else ckpt
)
model_s2.load_state_dict(state_dict, strict=True)
print(f"🔥 Successfully warmstarted 2v2 Learner Model from: {STAGE1_BEST}")


train_mappo(
    envs=envs_s2,
    model=model_s2,
    device=device,
    team_size=TEAM_SIZE,
    total_timesteps=20_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,  
    update_epochs=2, 
    minibatch_size=2048,
    lr_init=3e-5,  
    lr_final=3e-6,
    ent_coef_init=0.02,  
    ent_coef_final=0.006,
    gamma=0.996,
    gae_lambda=0.98,
    active_tiers=["heuristic"],
    target_tier="heuristic",
    filter_thresholds={
        "heuristic": 0.50,
    },
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    save_dir=SAVE_DIR_S2,
    pool_dir=POOL_DIR_S2,
    eval_episodes=50,  
    eval_freq=200_000,
    max_steps=ROUND_STEPS_S2,
)

envs_s2.close()

✅ Initialized Stage 2 pool with Stage 1 Champion weights.
🔥 Successfully warmstarted 2v2 Learner Model from: models/stage1/phase4/best_model.pt
🚀 MAPPO Initialized | Format: 2v2 | Envs: 24 | Step Batch: 12288 | Device: cuda

📊 [EVALUATION @ Step 208,896 | SPS: 4185 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  42.0% | Reward: +0.298 | Goals: 49 Scored, 35 Conceded (+14 Net)
   ❌ Retaining current baseline. Did not pass criteria for heuristic: [WR: None, Reward: None, Net: None]

📊 [EVALUATION @ Step 405,504 | SPS: 2794 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  50.0% | Reward: +0.762 | Goals: 50 Scored, 24 Conceded (+26 Net)
🏆 NEW CHAMPION REGISTERED @ step 405,504 -> models/stage2/phase1/pool/history_405504.pt
   ⭐⭐ PROMOTED! New Best Score (heuristic) -> [WR: 50.0%, Reward: +0.762, Net: +26]
      (Defeated previous record: [WR: None, Reward: None]) -> Saved: models/stage2/phase1/best_model.pt

📊 [EVALUATION @ Step 602,112 | SPS: 2493 | Tiers: ['heurist

In [13]:
import torch
from src.rl_2.visualization import evaluate_and_generate_html

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

evaluate_and_generate_html(
    red_agent="models/stage2/phase1/final_model.pt",
    blue_agent="heuristic",
    red_team_size=2,
    blue_team_size=3,
    device=device,
    filename="stage4_phase1_2vs3.html",
    num_episodes=10,
    max_steps=3600,
    base_seed=203,
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/training_2/render/stage4_phase1_2vs3.html


'render/stage4_phase1_2vs3.html'

# Phase 2

In [1]:
# Setup 2v3 Training against Counter-Attacking Bots
import os
import shutil
import sys
import gymnasium as gym
import torch

sys.path.insert(0, os.path.abspath(".."))

from src.rl_2.env_adapter import MatchEnv
from src.rl_2.model import ActorCritic
from src.rl_2.pool import PoolOpponentController
from src.rl_2.ppo import train_mappo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── 2v3 Setup ──
LEARNER_TEAM_SIZE = 2
OPP_TEAM_SIZE = 3

PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0
ROUND_STEPS_P2 = 3600  # 60-second regulation matches
NUM_ENVS = 16

SAVE_DIR_S2_P2 = "models/stage2/phase2"
POOL_DIR_S2_P2 = os.path.join(SAVE_DIR_S2_P2, "pool")
os.makedirs(POOL_DIR_S2_P2, exist_ok=True)

# Warmstart from your latest Phase 1 2v2 best model
phase1_best = "models/stage2/phase1/best_model.pt"
if os.path.exists(phase1_best):
  seed_weights = phase1_best
  print(f"🔥 Warmstarting Phase 2 from Phase 1 Best: {seed_weights}")
else:
  raise FileNotFoundError(f"Missing Phase 1 model at: {phase1_best}")

shutil.copy(seed_weights, os.path.join(POOL_DIR_S2_P2, "champion.pt"))
shutil.copy(seed_weights, os.path.join(POOL_DIR_S2_P2, "history_0.pt"))


def make_s2_p2_env(env_rank: int):

  def _thunk():
    # 90% Heuristic (GK, Defender, Target Striker), 10% Random
    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S2_P2,
        team="blue",
        device="cpu",
        p_random=0.10,
        p_heuristic=0.90,
    )
    env = MatchEnv(
        learner_team_size=LEARNER_TEAM_SIZE,
        opp_team_size=OPP_TEAM_SIZE,
        learner_team="red",
        max_round_steps=ROUND_STEPS_P2,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
    )
    env.reset(seed=8000 + env_rank)
    return env

  return _thunk


envs_s2_p2 = gym.vector.AsyncVectorEnv(
    [make_s2_p2_env(i) for i in range(NUM_ENVS)],
    context="fork",
)

model_s2_p2 = ActorCritic().to(device)
ckpt = torch.load(seed_weights, map_location=device, weights_only=False)
state_dict = (
    ckpt["model_state_dict"]
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt
    else ckpt
)
model_s2_p2.load_state_dict(state_dict, strict=True)
print(f"✅ Loaded weights into Model successfully.")

Using device: cuda
🔥 Warmstarting Phase 2 from Phase 1 Best: models/stage2/phase1/best_model.pt
✅ Loaded weights into Model successfully.


In [2]:
train_mappo(
    envs=envs_s2_p2,
    model=model_s2_p2,
    device=device,
    team_size=LEARNER_TEAM_SIZE,  # 2 RL learners in trajectory buffer
    opp_team_size=OPP_TEAM_SIZE,  # 3 bots in simulation
    total_timesteps=12_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    update_epochs=3,              # 3 epochs for stronger policy adjustment
    minibatch_size=1024,
    lr_init=3e-5,                 # Higher learning rate to break symmetrical reflexes
    lr_final=3e-6,
    ent_coef_init=0.010,
    ent_coef_final=0.002,
    gamma=0.996,
    gae_lambda=0.97,
    active_tiers=["heuristic"],   # Target the 3-bot coordinator
    target_tier="heuristic",
    filter_thresholds=None,      
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    save_dir=SAVE_DIR_S2_P2,
    pool_dir=POOL_DIR_S2_P2,
    eval_episodes=50,
    eval_freq=200_000,
    max_steps=ROUND_STEPS_P2,
)

envs_s2_p2.close()

🚀 MAPPO Initialized | Format: 2v2 | Envs: 16 | Step Batch: 8192 | Device: cuda

📊 [EVALUATION @ Step 204,800 | SPS: 3689 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  54.0% | Reward: +1.634 | Goals: 80 Scored, 18 Conceded (+62 Net)
🏆 NEW CHAMPION REGISTERED @ step 204,800 -> models/stage2/phase2/pool/history_204800.pt
   ⭐⭐ PROMOTED! New Best Score (heuristic) -> [WR: 54.0%, Reward: +1.634, Net: +62]
      (Defeated previous record: [WR: None, Reward: None]) -> Saved: models/stage2/phase2/best_model.pt

📊 [EVALUATION @ Step 401,408 | SPS: 2596 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  70.0% | Reward: +2.004 | Goals: 85 Scored, 18 Conceded (+67 Net)
🏆 NEW CHAMPION REGISTERED @ step 401,408 -> models/stage2/phase2/pool/history_401408.pt
   ⭐⭐ PROMOTED! New Best Score (heuristic) -> [WR: 70.0%, Reward: +2.004, Net: +67]
      (Defeated previous record: [WR: 54.0%, Reward: +1.634]) -> Saved: models/stage2/phase2/best_model.pt

📊 [EVALUATION @ Step 606,208 | 

In [5]:
from src.rl_2.visualization import evaluate_and_generate_html

evaluate_and_generate_html(
    red_agent=os.path.join(SAVE_DIR_S2_P2, "best_model.pt"),
    blue_agent="heuristic",
    red_team_size=2,
    blue_team_size=3,
    device=device,
    filename="stage2_phase2_2v3.html",
    num_episodes=10,
    max_steps=3600,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    goal_height=GOAL_H_REG,
    base_seed=1,
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/training_2/render/stage2_phase2_2v3.html


'render/stage2_phase2_2v3.html'

# Phase 3

In [1]:
import os
import shutil
import sys
import gymnasium as gym
import torch

sys.path.insert(0, os.path.abspath(".."))

from src.rl_2.env_adapter import MatchEnv
from src.rl_2.model import ActorCritic
from src.rl_2.pool import PoolOpponentController
from src.rl_2.ppo import train_mappo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Hyperparameters ──
LEARNER_TEAM_SIZE = 2
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0
ROUND_STEPS_P3 = 3600    # 60s matches (360 decisions @ 6 Hz)
NUM_ENVS = 20

SAVE_DIR_S2_P3 = "models/stage2/phase3"
POOL_DIR_S2_P3 = os.path.join(SAVE_DIR_S2_P3, "pool")
os.makedirs(POOL_DIR_S2_P3, exist_ok=True)

# Warmstart from your latest best checkpoint
phase2_best = "models/stage2/phase2/best_model.pt"
phase1_best = "models/stage2/phase1/best_model.pt"

if os.path.exists(phase2_best):
  seed_weights = phase2_best
elif os.path.exists(phase1_best):
  seed_weights = phase1_best
else:
  raise FileNotFoundError("No pre-trained model found to seed Phase 3.")

shutil.copy(seed_weights, os.path.join(POOL_DIR_S2_P3, "champion.pt"))
shutil.copy(seed_weights, os.path.join(POOL_DIR_S2_P3, "history_0.pt"))
print(f"🔥 Seeded Phase 3 pool using weights from: {seed_weights}")


def make_s2_p3_env(env_rank: int):

  def _thunk():
    # 5% Random, 25% Heuristic, 70% Self-Play
    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S2_P3,
        team="blue",
        device="cpu",
        p_random=0.05,
        p_heuristic=0.25,
    )
    env = MatchEnv(
        learner_team_size=LEARNER_TEAM_SIZE,
        opp_team_size=2,          # Defaults to 2, dynamically toggles 2v2/2v3 on reset
        learner_team="red",
        max_round_steps=ROUND_STEPS_P3,
        action_repeat=10,         # 6 Hz control frequency
        heuristic_accel=4000.0,   # Buffed heuristic motor output
        heuristic_kick=1500.0,    # Buffed clearance impulse
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
    )
    env.reset(seed=9000 + env_rank)
    return env

  return _thunk


envs_s2_p3 = gym.vector.AsyncVectorEnv(
    [make_s2_p3_env(i) for i in range(NUM_ENVS)],
    context="fork",
)

model_s2_p3 = ActorCritic().to(device)
ckpt = torch.load(seed_weights, map_location=device, weights_only=False)
state_dict = (
    ckpt["model_state_dict"]
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt
    else ckpt
)
model_s2_p3.load_state_dict(state_dict, strict=True)
print(f"✅ Model warmstarted successfully.")

Using device: cuda
🔥 Seeded Phase 3 pool using weights from: models/stage2/phase2/best_model.pt
✅ Model warmstarted successfully.


In [2]:
# ── Train Phase 3 ──
train_mappo(
    envs=envs_s2_p3,
    model=model_s2_p3,
    device=device,
    team_size=LEARNER_TEAM_SIZE,
    opp_team_size=2,
    total_timesteps=20_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    update_epochs=3,
    minibatch_size=1024,
    lr_init=5e-5,
    lr_final=5e-6,
    ent_coef_init=0.008,
    ent_coef_final=0.001,
    gamma=0.995,
    gae_lambda=0.97,
    active_tiers=["heuristic", "champion"],
    target_tier="champion",               # Dethrone self-play champions
    filter_thresholds={"heuristic": 0.65}, # Gatekeeper: must beat buffed heuristics >= 65%
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    save_dir=SAVE_DIR_S2_P3,
    pool_dir=POOL_DIR_S2_P3,
    eval_episodes=20,
    eval_freq=200_000,
    max_steps=ROUND_STEPS_P3,
)

envs_s2_p3.close()

🚀 MAPPO Initialized | Format: 2v2 | Envs: 20 | Step Batch: 10240 | Device: cuda

📊 [EVALUATION @ Step 204,800 | SPS: 3072 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR: 100.0% | Reward: +3.970 | Goals: 55 Scored, 1 Conceded (+54 Net)
   ⚔️  vs Champion  [TARGET] | WR:  40.0% | Reward: +0.115 | Goals: 16 Scored, 13 Conceded (+3 Net)
   ❌ Retaining current baseline. Did not pass criteria for champion: [WR: 40.0%, Reward: +0.115, Net: +3]

📊 [EVALUATION @ Step 409,600 | SPS: 2250 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR: 100.0% | Reward: +3.915 | Goals: 54 Scored, 1 Conceded (+53 Net)
   ⚔️  vs Champion  [TARGET] | WR:  30.0% | Reward: -0.195 | Goals: 14 Scored, 13 Conceded (+1 Net)
   ❌ Retaining current baseline. Did not pass criteria for champion: [WR: 30.0%, Reward: -0.195, Net: +1]

📊 [EVALUATION @ Step 604,160 | SPS: 2047 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR: 100.0% | Reward: +4.025 | Goals: 55 Sco

In [7]:
from src.rl_2.visualization import evaluate_and_generate_html

evaluate_and_generate_html(
    red_agent=os.path.join(SAVE_DIR_S2_P3, "best_model.pt"),
    blue_agent="models/stage2/phase2/best_model.pt",
    red_team_size=2,
    blue_team_size=2,
    device=device,
    filename="stage2_phase3_2v2.html",
    num_episodes=20,
    max_steps=3600,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    goal_height=GOAL_H_REG,
    base_seed=1,
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/training_2/render/stage2_phase3_2v2.html


'render/stage2_phase3_2v2.html'